# 3D Anomaly Detection Benchmark — Anomaly-ShapeNet + Real3D-AD

**Setup:**
1. Enable the **T4 GPU** accelerator (Settings → Accelerator)
2. Attach the two datasets as Inputs (Add Input → Your Work → your uploads) —
   the roots are hard-coded in the *dataset* cell below
3. Set `REPO_URL` to your GitHub fork of this project
4. *Run All*

Expected wall-clock: Anomaly-ShapeNet ~30–60 min · Real3D-AD ~2–4 h.
For full-benchmark runs use *Save Version → Save & Run All* so everything
persists even if you disconnect.

In [ ]:
%pip install -q open3d
import torch
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ---- get the ad3d codebase -------------------------------------------------
import os, glob

REPO_URL = "https://github.com/YOUR_USERNAME/ad3d.git"   # <-- EDIT: your fork

if not os.path.exists('ad3d'):
    if 'YOUR_USERNAME' not in REPO_URL:
        os.system(f'git clone --depth 1 {REPO_URL}')
    if not os.path.exists('ad3d'):
        # fallback: the folder was uploaded as a Kaggle Dataset (Add Input)
        hits = glob.glob('/kaggle/input/**/run.py', recursive=True)
        assert hits, 'Set REPO_URL to your GitHub fork, or upload the ad3d folder as a Kaggle input.'
        os.symlink(os.path.dirname(hits[0]), 'ad3d')
print('code at:', os.path.abspath('ad3d'))

In [ ]:
# ---- dataset roots (Kaggle inputs) + verification + loader sanity -----------
import os, glob, sys

SHAPENET_ROOT = "/kaggle/input/datasets/smnihalahmed/anomaly-shapenet-v2/Anomaly-ShapeNet-v2/dataset/pcd"
REAL_ROOT     = "/kaggle/input/datasets/smnihalahmed/real3d-ad-pcd/Real3D-AD-PCD"

def verify(root, cat, name):
    n_cat = len([d for d in glob.glob(f"{root}/*/") if os.path.isdir(d)])
    tr = len(glob.glob(f"{root}/{cat}/train/*.pcd"))
    te = len(glob.glob(f"{root}/{cat}/test/*.pcd"))
    gt = len(glob.glob(f"{root}/{cat}/GT/*.txt")) or len(glob.glob(f"{root}/{cat}/gt/*.txt"))
    print(f"{name}: {n_cat} categories | {cat}: train={tr}  test={te}  gt={gt}")
    print(f"   -> {root}")
    assert n_cat > 0 and tr > 0 and te > 0 and gt > 0, f"{name} structure check failed"

verify(SHAPENET_ROOT, "ashtray0", "Anomaly-ShapeNet")
verify(REAL_ROOT, "airplane", "Real3D-AD")

# exercise the actual loaders on one category each (checks file formats too)
sys.path.insert(0, "ad3d/src")
from ad3d.datasets import load_train, load_test
for root, ds, cat in [(SHAPENET_ROOT, "shapenet", "ashtray0"),
                      (REAL_ROOT, "real3d", "airplane")]:
    tr = load_train(root, ds, cat)
    te = load_test(root, ds, cat)
    n_norm = sum(1 for s in te if s.label == 0)
    print(f"[loader] {ds}/{cat}: train={len(tr)}  test={len(te)} "
          f"({n_norm} normal / {len(te)-n_norm} anomalous)")

In [ ]:
# ---- smoke test: 2 categories, small settings (a few minutes) ---------------
SMOKE = ('--num-group 512 --group-size 64 --max-nn 60 --n-scales 2 '
         '--points-budget 30000')
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --classes ashtray0,bowl0 {SMOKE} --tag smoke

In [ ]:
# ---- FULL Anomaly-ShapeNet benchmark (40 categories) -----------------------
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --tag simple3dlite

In [ ]:
# ---- FULL Real3D-AD benchmark (12 categories) -------------------------------
!python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" --tag simple3dlite

In [ ]:
# ---- Real3D-AD ablation grid: alignment / cuts / topk -----------------------
# Targets the 360-degree-train vs single-view-test gap. Each run is tagged and
# compared against the simple3dlite baseline. Runtime note: --align icp adds a
# RANSAC+ICP registration per test cloud (expect ~2x the baseline runtime).
GRID = [
    ("r1-cuts4",            "--cuts 4"),
    ("r2-icp",              "--align icp"),
    ("r3-icp-cuts4",        "--align icp --cuts 4"),
    ("r4-icp-cuts4-topk32", "--align icp --cuts 4 --topk 32"),
    ("r5-topk32",           "--topk 32"),
]
for tag, extra in GRID:
    print(f"\n########## {tag}: {extra} ##########", flush=True)
    get_ipython().system(f'python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" {extra} --tag {tag}')

In [ ]:
# ---- summarize --------------------------------------------------------------
import pandas as pd, glob
frames = []
for f in sorted(glob.glob('results/*.csv')):
    df = pd.read_csv(f, index_col=0)
    df.index.name = 'category'
    df['source'] = f
    frames.append(df)
if frames:
    full = pd.concat(frames)
    means = full.groupby('source')[['o_auroc','p_auroc','p_aupr','p_auroc_pooled','p_aupr_pooled']].mean()
    display(means)
    full.drop(columns='source').to_csv('ad3d_results_summary.csv')
    print('saved ad3d_results_summary.csv')
else:
    print('no full-run results yet')

## Next steps

Open **`ad3d/NOVELTY_ROADMAP.md`** — it maps the open failure modes in these
benchmarks (point-level AUPR, the gemstone/starfish/duck categories, the
360°-train vs single-view-test gap) to the exact functions to modify:

| To change... | Edit... |
|---|---|
| features / descriptors | `src/ad3d/features.py` → `compute_fpfh_ms()` |
| grouping / aggregation | `src/ad3d/features.py` → `lfsa()` |
| memory bank / distance | `src/ad3d/memory.py` |
| scoring rules | `src/ad3d/scoring.py` |
| whole pipeline | `src/ad3d/method.py` → subclass `Simple3DLite` |

Every run writes its full config + per-category results into `results/*.json`,
so tagged comparisons are always reproducible.